In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import make_scorer, precision_score, ConfusionMatrixDisplay, confusion_matrix
from sklearn.decomposition import PCA
from sklearn import model_selection
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import geopandas as gpd
import scipy as sp
import folium


In [ ]:
tree_data = pd.read_csv(r'..\Data\new_york_tree_census_2015.csv')

In [ ]:
tree_data_small = tree_data[['tree_dbh', 'curb_loc', 'health', 'spc_common', 'sidewalk', 'zipcode']].copy()
tree_data_small

In [ ]:
# initial clean of the data - drop na
print("tree dataset rows total: ", len(tree_data_small))
tree_data_small.dropna(inplace=True)
print("tree dataset rows after na drop: ", len(tree_data_small))

tree_data_small.info()

In [ ]:
def get_ratio(series):
       val_counts = series.value_counts()
       if len(val_counts) > 1:
              return val_counts.iloc[0]/ val_counts.iloc[1]
       # if there is only one value then the ratio is 1
       return 1
           
def get_first(series):
       val_counts = series.value_counts()
       return val_counts.index[0]

def get_second(series):
       val_counts = series.value_counts()
       if len(val_counts) > 1:
              return val_counts.index[1]
       else:
              return val_counts.index[-1]

def get_third(series):
       val_counts = series.value_counts()
       if len(val_counts) > 2:
              return val_counts.index[2]
       else:
              return val_counts.index[-1]
       
def count_unique_vals(series):
       return series.value_counts().to_dict()
    
       

In [ ]:
# because we are going to join on zip code, we need to make meaningful aggregates for each column per zip code
tree_data_agg = tree_data_small.groupby('zipcode').agg(
       dbh_mean=('tree_dbh', 'mean'),
       curb_ratio=('curb_loc', get_ratio),
       sidewalk_ratio=('sidewalk', get_ratio),
       health_counts=('health', count_unique_vals),
       species_1=('spc_common', get_first),
       species_2=('spc_common', get_second),
       species_3=('spc_common', get_third),
       tree_count=('tree_dbh', 'size')
    
).reset_index()

health_counts_df = pd.DataFrame(tree_data_agg['health_counts'].tolist()).fillna(0).astype(int)
tree_data_agg = pd.concat([tree_data_agg, health_counts_df], axis=1).drop(columns=['health_counts'])

tree_data_agg


In [ ]:
# need to encode categorical data
encoder = LabelEncoder()

# encode the three species columns
tree_encoded_df = tree_data_agg.copy()
for col in ['species_1', 'species_2', 'species_3']:
    tree_encoded_df[col] = encoder.fit_transform(tree_data_agg[col])

tree_encoded_df

In [ ]:
crime_data = pd.read_csv(r'..\Data\2015_Crime.csv', low_memory=False)

In [ ]:
crime_data.head()
# Keep only the specified columns
crime_data = crime_data[['Violation Date', 'Violation Time', 'Issuing Agency',
                         'Violation Location (Zip Code)', 
                         'Penalty Imposed', 'Charge #1: Code', 
                         'Charge #2: Code', 'Charge #3: Code', 'Charge #4: Code', 
                         'Charge #5: Code', 'Charge #6: Code', 'Charge #7: Code', 
                         'Charge #8: Code', 'Charge #9: Code', 'Charge #10: Code']]

# Remove rows where the 'Violation Location (Zip Code)' column is NaN
# print("crime dataset rows total: ", len(crime_data))
# crime_data = crime_data.dropna(subset=['Violation Location (Zip Code)'])
# 
# crime_data = crime_data.dropna(subset=['Issuing Agency'])
# print("crime dataset rows after na drop: ", len(crime_data))

crime_data.head()

In [ ]:
# Extract columns containing charges
charge_columns = [col for col in crime_data.columns if "Charge" in col]
print("crime data len pre drop pre melt: ", len(crime_data))

crime_data = crime_data.melt(
    id_vars=[col for col in crime_data.columns if col not in charge_columns],  # Keep these columns unchanged  # Columns to unpivot
    var_name="Original Charge Column",  # New column for original charge column names
    value_name="Charge: Code",  # New column for charge values
)

# Drop rows where Charge: Code is NaN
# crime_data = crime_data.dropna(subset=["Charge: Code"]).drop(columns=["Original Charge Column"])
print("crime data len pre drop: ", len(crime_data))
crime_data.dropna(inplace=True)
print("crime data len post drop: ", len(crime_data))

# Reset index for a clean DataFrame
crime_data = crime_data.reset_index(drop=True)

# because we are going to join on zip code, we need to make meaningful aggregates for each column per zip code
crime_data_final = crime_data.groupby('Violation Location (Zip Code)').agg(
    issuing_agency_1=('Issuing Agency', get_first),
    issuing_agency_2=('Issuing Agency', get_second),
    issuing_agency_3=('Issuing Agency', get_third),
    penalty_imposed=('Penalty Imposed', 'mean'),
    charge_1 = ('Charge: Code', get_first),
    charge_2 = ('Charge: Code', get_second),
    charge_3 = ('Charge: Code', get_third),
    crime_count = ('Issuing Agency', 'size')
).reset_index()

# calculating if the crime count is above the mean
crime_data_final['crime_above_avg'] = (crime_data_final['crime_count'] > crime_data_final['crime_count'].mean()).astype(int)

crime_data_final.rename(columns={'Violation Location (Zip Code)': 'zipcode'}, inplace=True)

crime_data_final.drop(crime_data_final[~crime_data_final['zipcode'].astype(str).apply(lambda x: x.isdigit())].index, inplace=True)

crime_data_final['zipcode'] = crime_data_final['zipcode'].astype(int)

crime_data_final.head()

In [ ]:
encoder = LabelEncoder()

# encode the three species columns
crime_encoded_df = crime_data_final.copy()
for col in ['issuing_agency_1', 'issuing_agency_2', 'issuing_agency_3']:
    crime_encoded_df[col] = encoder.fit_transform(crime_data_final[col])

for col in ['charge_1', 'charge_2', 'charge_3']:
    crime_encoded_df[col] = encoder.fit_transform(crime_data_final[col])


In [ ]:
crime_encoded_df

In [ ]:
# combining datasets for ML algs
tree_final = tree_encoded_df
crime_final = crime_encoded_df

merged_df = pd.merge(tree_final, crime_final, on='zipcode', how='inner')
merged_df.set_index('zipcode', inplace=True)

# visualize data
sns.scatterplot(x='tree_count', y='penalty_imposed', hue='crime_above_avg', data=merged_df)
sns.regplot(x='tree_count', y='penalty_imposed', scatter=False, color='red', data=merged_df)
r, p = sp.stats.pearsonr(merged_df['tree_count'], merged_df['penalty_imposed'])
plt.ylabel('Crime Penalty Imposed (Dollars)')
plt.xlabel('Tree Count')
plt.title('Crime Penalty Imposed in Dollars vs Tree Count per NYC Zip Code')
plt.annotate("r = {:.3f}".format(r), (1, 2300))
plt.show()

sns.scatterplot(x='dbh_mean', y='crime_count', hue='crime_above_avg', data=merged_df)
sns.regplot(x='dbh_mean', y='crime_count', scatter=False, color='red', data=merged_df)
r, p = sp.stats.pearsonr(merged_df['dbh_mean'], merged_df['crime_count'])
plt.ylabel('Crime Count')
plt.xlabel('Mean Tree DBH')
plt.title('Tree Mean DBH vs Crime Count per NYC Zip Code')
plt.annotate("r = {:.3f}".format(r), (5, 13000))
plt.show()

sns.scatterplot(x='tree_count', y='crime_count', hue='crime_above_avg', data=merged_df)
sns.regplot(x='tree_count', y='crime_count', scatter=False, color='red', data=merged_df)
r, p = sp.stats.pearsonr(merged_df['tree_count'], merged_df['crime_count'])
plt.ylabel('Crime Count')
plt.xlabel('Tree Count')
plt.title('Tree Count vs Crime Count per NYC Zip Code')
plt.annotate("r = {:.3f}".format(r), (0, 13000))
plt.show()


In [ ]:
to_scale_df = merged_df.reset_index()

# scaled data
non_scaling_cols = ['crime_above_avg', 'zipcode']
scaling_cols = [col for col in to_scale_df.columns if col not in non_scaling_cols]

scaler = StandardScaler()
to_scale_df[scaling_cols] = pd.DataFrame(scaler.fit_transform(to_scale_df[scaling_cols]))
scaled_df = to_scale_df[scaling_cols + non_scaling_cols]

scaled_df['zipcode'] = to_scale_df['zipcode']
scaled_df.set_index('zipcode', inplace=True)


In [ ]:
# ML algs - predicting crime count 

# train test split
X = scaled_df.drop(['crime_above_avg', 'crime_count'], axis=1)
y = scaled_df['crime_above_avg']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=1/3.) 

# Finding the optimal k 
print("- Testing Different Values of k -") 
k_range = range(2, 21)  # Test k from 1 to 10

results = []


for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k) 
    knn.fit(X_train, y_train)
    precision_scorer = make_scorer(precision_score, average='weighted', zero_division=0) 
    scores = model_selection.cross_validate(knn, X_test, y_test, cv=5,
                                             scoring={'precision_weighted': precision_scorer,
                                                      'f1_weighted': 'f1_weighted',
                                                      'recall_weighted': 'recall_weighted'})

    results.append({
        'k': k,
        'f1_weighted': np.mean(scores['test_f1_weighted']), 
        'precision_weighted': np.mean(scores['test_precision_weighted']), 
        'recall_weighted': np.mean(scores['test_recall_weighted'])
    })

# Convert results to DataFrame 
results_df = pd.DataFrame(results) 

# Find the best k 
best_k = results_df.loc[results_df['f1_weighted'].idxmax(), 'k'] 
print(f"Best k: {best_k}") 

# Visualize performance metrics vs. k 
plt.figure(figsize=(10, 6))
plt.plot(results_df['k'], results_df['f1_weighted'], label='F1-Score', marker='o')
plt.plot(results_df['k'], results_df['precision_weighted'], label='Precision', marker='o')
plt.plot(results_df['k'], results_df['recall_weighted'], label='Recall', marker='o')
plt.xlabel('Number of Neighbors (k)')
plt.ylabel('Score')
plt.title('Performance Metrics vs. Number of Neighbors')
plt.legend()
plt.grid(True)
plt.show()

# Train the model using the best k 
knn = KNeighborsClassifier(n_neighbors=best_k) 
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test) 

print("- 5-Fold Cross Validation -") 

precision_scorer = make_scorer(precision_score, average='weighted', zero_division=0) 

scores = model_selection.cross_validate(knn, X_test, y_test, cv=5,
                                        scoring={'precision_weighted': precision_scorer,  
                                                 'f1_weighted': 'f1_weighted',  
                                                 'recall_weighted': 'recall_weighted'}) 

print("F1: ", f"{np.mean(scores['test_f1_weighted']):.4f}") 
print("Precision: ", f"{np.mean(scores['test_precision_weighted']):.4f}") 
print("Recall: ", f"{np.mean(scores['test_recall_weighted']):.4f}") 



# Compute confusion matrix 
cm = confusion_matrix(y_test, y_pred, labels=knn.classes_) 

# Display confusion matrix 

disp = ConfusionMatrixDisplay(confusion_matrix=cm) 
disp.plot(cmap='Blues') 
plt.title("Confusion Matrix") 
plt.show() 

In [ ]:
# k means
inertia = []
k_values = range(1, 11)

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_train)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(k_values, inertia, marker='o')
plt.title('Elbow Method for Optimal Clusters')
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')
plt.show()

optimal_clusters = 3  

kmeans = KMeans(n_clusters=optimal_clusters)
cluster_df = scaled_df.copy()
cluster_df['cluster'] = kmeans.fit_predict(cluster_df)

print(cluster_df.groupby('cluster').mean())

sns.scatterplot(
    x=cluster_df.groupby('cluster')['Good'].mean(),
    y=cluster_df.groupby('cluster')['crime_count'].mean(),
    hue=cluster_df.groupby('cluster').size().index, 
    palette='viridis',
    s=100,
    legend='full'
)
plt.title('Crime Count vs. Good Tree Health Across Clusters')
plt.xlabel('Average Number of Good Trees')
plt.ylabel('Average Crime Count')
plt.legend(title='Cluster')
plt.grid()
plt.show()

plt.figure(figsize=(12, 6))
sns.scatterplot(
    x=cluster_df.groupby('cluster')['Fair'].mean(),
    y=cluster_df.groupby('cluster')['crime_count'].mean(),
    hue=cluster_df.groupby('cluster').size().index,  
    palette='coolwarm',
    s=100,
    legend='full'
)
plt.title('Crime Count vs. Fair Tree Health Across Clusters')
plt.xlabel('Average Number of Fair Trees')
plt.ylabel('Average Crime Count')
plt.legend(title='Cluster')
plt.grid()
plt.show()

plt.figure(figsize=(12, 6))
sns.scatterplot(
    x=cluster_df.groupby('cluster')['Poor'].mean(),
    y=cluster_df.groupby('cluster')['crime_count'].mean(),
    hue=cluster_df.groupby('cluster').size().index,  
    palette='plasma',
    s=100,
    legend='full'
)
plt.title('Crime Count vs. Poor Tree Health Across Clusters')
plt.xlabel('Average Number of Poor Trees')
plt.ylabel('Average Crime Count')
plt.legend(title='Cluster')
plt.grid()
plt.show()

In [ ]:
# pca
pca = PCA(n_components=2)
pca_components = pca.fit_transform(cluster_df.drop(columns=['cluster']))

plt.figure(figsize=(10, 6))
for cluster in cluster_df['cluster'].unique():
    cluster_points = pca_components[cluster_df['cluster'] == cluster]
    plt.scatter(cluster_points[:, 0], cluster_points[:, 1], label=f'Cluster {cluster}', alpha=0.7)
    
plt.title('K-Means Clustering Visualization (2D PCA Projection)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend()
plt.grid(True)
plt.show()

for col in merged_df.columns:
    if merged_df[col].dtype == 'datetime64[ns]' or isinstance(merged_df[col].iloc[0], pd.Timestamp):
        merged_df[col] = merged_df[col].astype(str)  
    elif merged_df[col].dtype == 'object':
        merged_df[col] = merged_df[col].astype(str)  

In [ ]:
# map analysis
geojson_path = 'nyc-zip-code-tabulation-areas-polygons.geojson'  
geo_data = gpd.read_file(geojson_path)

geo_data = geo_data.explode(index_parts=False)
geo_data['postalCode'] = geo_data['postalCode'].astype(str)

zipcode_df = merged_df.reset_index()

zipcode_df['zipcode'] = zipcode_df['zipcode'].astype(str)

merged_geo = geo_data.set_index('postalCode').join(zipcode_df.set_index('zipcode'))

centroid = geo_data.geometry.centroid.unary_union.centroid
map_center = [centroid.y, centroid.x] 

crime_map = folium.Map(location=map_center, zoom_start=10)  
folium.Choropleth(
    geo_data=merged_geo.__geo_interface__,
    data=zipcode_df,
    columns=['zipcode', 'crime_count'],
    key_on='feature.id',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Crime Count'
).add_to(crime_map)
crime_map.save('crime_count_heatmap.html')  

tree_map = folium.Map(location=map_center, zoom_start=10) 
folium.Choropleth(
    geo_data=merged_geo.__geo_interface__,
    data=zipcode_df,
    columns=['zipcode', 'dbh_mean'],
    key_on='feature.id',
    fill_color='BuGn',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Average Tree Diameter (dbh_mean)'
).add_to(tree_map)
tree_map.save('dbh_mean_heatmap.html') 

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Train a Decision Tree Classifier
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

# Feature importance
importances = dt.feature_importances_
features = X.columns

# Create a DataFrame for better readability
feature_importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(feature_importance_df)

# Plot feature importances
plt.figure(figsize=(10, 6))
plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'])
plt.xlabel('Feature Importance')
plt.ylabel('Feature')
plt.title('Feature Importance using Decision Tree Classifier')
plt.gca().invert_yaxis()  # Invert y-axis for better readability
plt.show()
